# GAN ablations for NeuralNector report (Colab-friendly)

## Experiment plan (designed for **under ~1 hour** total on a typical Colab GPU)

Full Flowers training in `GAN_pytorch.ipynb` uses **3000 epochs**; here we use **short budgets** to compare *relative* trends (loss curves, qualitative samples), not to reproduce final production quality. State that clearly in your report.

| ID | Dataset | What changes | Rationale |
|----|---------|--------------|------------|
| `M1_baseline` | MNIST | Default: label smooth 0.1, lr\_G=2×lr, lr\_D=0.5×lr, skip D if d\_loss&lt;0.6, extra G if d\_loss&lt;0.4 | Reference |
| `M2_no_label_smooth` | MNIST | `label_smooth=0` | Harsher D targets; often less stable |
| `M3_symmetric_lr` | MNIST | lr\_G = lr\_D = base lr | D may dominate |
| `M4_always_train_D` | MNIST | Never skip D (threshold -1) | No adaptive pause |
| `M5_latent_50` | MNIST | `latent_dim=50` | Capacity vs diversity |
| `M6_beta1_0.9` | MNIST | Adam $\beta_1=0.9$ vs 0.5 | Common GAN ablation |
| `F1_flowers_baseline` | Flowers-102 | Default + augmentation | Match prod recipe (short) |
| `F2_flowers_no_aug` | Flowers-102 | No augmentation | Less diversity regularization |

**Runtime knobs** (cell below): reduce `MNIST_EPOCHS` / `FLOWERS_EPOCHS` if you hit the Colab limit.

## What to paste back for the report
After **Run all**, copy:
1. The printed **`EXPERIMENTS_JSON`** block (entire JSON array).
2. Optionally attach / describe the saved **`experiment_grids/`** PNGs.
3. Upload **`experiments_summary.csv`** from the Colab file browser if you want table numbers checked.

The JSON includes: `experiment_id`, dataset, hyperparameters, `final_g_loss`, `final_d_loss`, `mean_g_loss_last_3_epochs`, `mean_d_loss_last_3_epochs`, `seconds`, `hardware_note`.

In [ ]:
%pip install -q pytorch-lightning

In [ ]:
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from pytorch_lightning.callbacks import Callback
from torch.utils.data import ConcatDataset, DataLoader, random_split
from torchvision.datasets import Flowers102, MNIST

random_seed = 42
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)
torch.set_float32_matmul_precision("medium")

# --- budget: tune down if needed ---
MNIST_EPOCHS = 12
FLOWERS_EPOCHS = 10
BATCH_MNIST = 128
BATCH_FLOWERS = 64
# Colab: 0 or 2; try 0 if dataloader is slow
NUM_WORKERS = 2

DATA_DIR = "./data"
OUT_DIR = Path("experiment_outputs")
GRID_DIR = OUT_DIR / "experiment_grids"
OUT_DIR.mkdir(exist_ok=True)
GRID_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", GPU_NAME)

In [ ]:
class MNISTDataModule(pl.LightningDataModule):
    def __init__(self, data_dir=DATA_DIR, batch_size=BATCH_MNIST, num_workers=NUM_WORKERS):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.transform = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        )

    def prepare_data(self):
        MNIST(self.data_dir, train=True, download=True)
        MNIST(self.data_dir, train=False, download=True)

    def setup(self, stage=None):
        if stage == "fit" or stage is None:
            mnist_full = MNIST(self.data_dir, train=True, transform=self.transform)
            self.mnist_train, self.mnist_val = random_split(
                mnist_full, [55000, 5000], generator=torch.Generator().manual_seed(random_seed)
            )

    def train_dataloader(self):
        return DataLoader(
            self.mnist_train,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            persistent_workers=bool(self.num_workers),
        )


class FlowersDataModule(pl.LightningDataModule):
    def __init__(
        self,
        data_dir=DATA_DIR,
        batch_size=BATCH_FLOWERS,
        num_workers=NUM_WORKERS,
        image_size=64,
        combine_train_val=True,
        use_augmentation=True,
    ):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.image_size = image_size
        self.combine_train_val = combine_train_val
        self.use_augmentation = use_augmentation

        base_transform = [
            transforms.Resize(image_size),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]
        if use_augmentation:
            self.train_transform = transforms.Compose(
                [
                    transforms.Resize(int(image_size * 1.1)),
                    transforms.RandomCrop(image_size),
                    transforms.RandomHorizontalFlip(p=0.5),
                    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                    transforms.ToTensor(),
                    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                ]
            )
        else:
            self.train_transform = transforms.Compose(base_transform)

    def prepare_data(self):
        Flowers102(self.data_dir, split="train", download=True)
        Flowers102(self.data_dir, split="val", download=True)
        Flowers102(self.data_dir, split="test", download=True)

    def setup(self, stage=None):
        if stage == "fit" or stage is None:
            flowers_train = Flowers102(self.data_dir, split="train", transform=self.train_transform)
            if self.combine_train_val:
                flowers_val = Flowers102(self.data_dir, split="val", transform=self.train_transform)
                self.flowers_train = ConcatDataset([flowers_train, flowers_val])
            else:
                self.flowers_train = flowers_train

    def train_dataloader(self):
        return DataLoader(
            self.flowers_train,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            persistent_workers=bool(self.num_workers),
        )

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, channels=1, image_size=28):
        super().__init__()
        self.channels = channels
        self.image_size = image_size
        if image_size == 64:
            self.conv1 = nn.Conv2d(channels, 64, kernel_size=4, stride=2, padding=1)
            self.conv2 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
            self.bn2 = nn.BatchNorm2d(128)
            self.conv3 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)
            self.bn3 = nn.BatchNorm2d(256)
            self.conv4 = nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1)
            self.bn4 = nn.BatchNorm2d(512)
            self.fc = nn.Linear(4 * 4 * 512, 1)
        else:
            flattened_size = 4 * 4 * 20
            self.conv1 = nn.Conv2d(channels, 10, kernel_size=5)
            self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
            self.conv2_drop = nn.Dropout2d()
            self.fc1 = nn.Linear(flattened_size, 50)
            self.fc2 = nn.Linear(50, 1)

    def forward(self, x):
        if self.image_size == 64:
            x = F.leaky_relu(self.conv1(x), 0.2)
            x = F.leaky_relu(self.bn2(self.conv2(x)), 0.2)
            x = F.leaky_relu(self.bn3(self.conv3(x)), 0.2)
            x = F.leaky_relu(self.bn4(self.conv4(x)), 0.2)
            x = x.view(x.size(0), -1)
            x = self.fc(x)
        else:
            x = F.relu(F.max_pool2d(self.conv1(x), 2))
            x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
            x = x.view(x.size(0), -1)
            x = F.relu(self.fc1(x))
            x = F.dropout(x, training=self.training)
            x = self.fc2(x)
        return torch.sigmoid(x)


class Generator(nn.Module):
    def __init__(self, latent_dim, channels=1, image_size=28):
        super().__init__()
        self.image_size = image_size
        if image_size == 64:
            self.lin1 = nn.Linear(latent_dim, 4 * 4 * 512)
            self.ct1 = nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1)
            self.bn1 = nn.BatchNorm2d(256)
            self.ct2 = nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1)
            self.bn2 = nn.BatchNorm2d(128)
            self.ct3 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
            self.bn3 = nn.BatchNorm2d(64)
            self.ct4 = nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1)
            self.bn4 = nn.BatchNorm2d(32)
            self.conv = nn.Conv2d(32, channels, kernel_size=3, padding=1)
        else:
            self.lin1 = nn.Linear(latent_dim, 7 * 7 * 64)
            self.ct1 = nn.ConvTranspose2d(64, 32, 4, stride=2)
            self.ct2 = nn.ConvTranspose2d(32, 16, 4, stride=2)
            self.conv = nn.Conv2d(16, channels, kernel_size=7)

    def forward(self, x):
        x = self.lin1(x)
        x = F.relu(x)
        if self.image_size == 64:
            x = x.view(-1, 512, 4, 4)
            x = F.relu(self.bn1(self.ct1(x)))
            x = F.relu(self.bn2(self.ct2(x)))
            x = F.relu(self.bn3(self.ct3(x)))
            x = F.relu(self.bn4(self.ct4(x)))
            x = torch.tanh(self.conv(x))
        else:
            x = x.view(-1, 64, 7, 7)
            x = F.relu(self.ct1(x))
            x = F.relu(self.ct2(x))
            x = torch.tanh(self.conv(x))
        return x


class GANExperiment(pl.LightningModule):
    """Same logic as GAN_pytorch / models.py; extra hparams for ablations."""

    def __init__(
        self,
        latent_dim=100,
        channels=1,
        image_size=28,
        lr=0.0002,
        label_smooth=0.1,
        lr_g_mult=2.0,
        lr_d_mult=0.5,
        beta1=0.5,
        # skip D when d_loss < this (set -1.0 to never skip)
        d_skip_threshold=0.6,
        # second G step when d_loss < this (set -1.0 to disable)
        d_double_g_threshold=0.4,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.automatic_optimization = False
        self.generator = Generator(latent_dim=latent_dim, channels=channels, image_size=image_size)
        self.discriminator = Discriminator(channels=channels, image_size=image_size)
        self.validation_z = torch.randn(8, latent_dim)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
                nn.init.normal_(m.weight, 0.0, 0.02)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, z):
        return self.generator(z)

    def adversarial_loss(self, y_hat, y):
        return F.binary_cross_entropy(y_hat, y)

    def training_step(self, batch, batch_idx):
        opt_g, opt_d = self.optimizers()
        real_imgs, _ = batch
        batch_size = real_imgs.size(0)
        z = torch.randn(batch_size, self.hparams.latent_dim, device=real_imgs.device)

        opt_d.zero_grad()
        y_hat_real = self.discriminator(real_imgs)
        y_real = torch.ones(batch_size, 1, device=real_imgs.device) * (
            1.0 - self.hparams.label_smooth
        )
        real_loss = self.adversarial_loss(y_hat_real, y_real)
        fake_imgs_det = self(z).detach()
        y_hat_fake = self.discriminator(fake_imgs_det)
        y_fake = torch.zeros(batch_size, 1, device=real_imgs.device) + self.hparams.label_smooth
        fake_loss = self.adversarial_loss(y_hat_fake, y_fake)
        d_loss = (real_loss + fake_loss) / 2

        thr_skip = self.hparams.d_skip_threshold
        if thr_skip is None or thr_skip < 0 or d_loss.item() >= thr_skip:
            self.manual_backward(d_loss)
            opt_d.step()

        opt_g.zero_grad()
        fake_imgs = self(z)
        y_hat = self.discriminator(fake_imgs)
        y = torch.ones(batch_size, 1, device=real_imgs.device)
        g_loss = self.adversarial_loss(y_hat, y)
        self.manual_backward(g_loss)
        opt_g.step()

        thr_dg = self.hparams.d_double_g_threshold
        if thr_dg is not None and thr_dg >= 0 and d_loss.item() < thr_dg:
            opt_g.zero_grad()
            z2 = torch.randn(batch_size, self.hparams.latent_dim, device=real_imgs.device)
            fake2 = self(z2)
            y_hat2 = self.discriminator(fake2)
            g_loss2 = self.adversarial_loss(y_hat2, y)
            self.manual_backward(g_loss2)
            opt_g.step()
            g_loss = (g_loss + g_loss2) / 2

        self.log("g_loss", g_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("d_loss", d_loss, on_step=False, on_epoch=True, prog_bar=True)
        return {"loss": g_loss + d_loss}

    def configure_optimizers(self):
        lr_g = self.hparams.lr * self.hparams.lr_g_mult
        lr_d = self.hparams.lr * self.hparams.lr_d_mult
        b1 = self.hparams.beta1
        opt_g = torch.optim.Adam(self.generator.parameters(), lr=lr_g, betas=(b1, 0.999))
        opt_d = torch.optim.Adam(self.discriminator.parameters(), lr=lr_d, betas=(b1, 0.999))
        return [opt_g, opt_d], []


class LossHistoryCallback(Callback):
    def __init__(self):
        self.g_loss_epochs = []
        self.d_loss_epochs = []

    def on_train_epoch_end(self, trainer, pl_module):
        g = trainer.callback_metrics.get("g_loss")
        d = trainer.callback_metrics.get("d_loss")
        if g is not None:
            self.g_loss_epochs.append(float(g.detach().cpu() if hasattr(g, "detach") else g))
        if d is not None:
            self.d_loss_epochs.append(float(d.detach().cpu() if hasattr(d, "detach") else d))


def save_sample_grid(model, path: Path, n_show: int = 8):
    model.eval()
    dev = next(model.parameters()).device
    z = model.validation_z.to(dev)
    with torch.no_grad():
        samples = model(z).cpu()
    samples = (samples + 1) / 2.0
    samples = torch.clamp(samples, 0, 1)
    c = model.hparams.channels
    s = model.hparams.image_size
    fig, axes = plt.subplots(2, 4, figsize=(8, 4))
    for i, ax in enumerate(axes.flat):
        if i >= n_show:
            break
        if c == 1:
            ax.imshow(samples[i, 0].numpy(), cmap="gray", interpolation="none")
        else:
            ax.imshow(samples[i].permute(1, 2, 0).numpy(), interpolation="none")
        ax.axis("off")
    plt.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    model.train()

In [ ]:
def run_experiment(
    experiment_id: str,
    dataset: str,
    max_epochs: int,
    model_kwargs: dict,
    flowers_use_augmentation: bool = True,
):
    assert dataset in ("mnist", "flowers")
    pl.seed_everything(random_seed, workers=True)

    if dataset == "mnist":
        dm = MNISTDataModule(batch_size=BATCH_MNIST, num_workers=NUM_WORKERS)
        ch, sz = 1, 28
    else:
        dm = FlowersDataModule(
            batch_size=BATCH_FLOWERS,
            num_workers=NUM_WORKERS,
            use_augmentation=flowers_use_augmentation,
        )
        ch, sz = 3, 64

    model = GANExperiment(channels=ch, image_size=sz, **model_kwargs)
    hist = LossHistoryCallback()

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        devices=1,
        enable_checkpointing=False,
        logger=False,
        enable_progress_bar=True,
        log_every_n_steps=5,
        callbacks=[hist],
    )

    t0 = time.perf_counter()
    trainer.fit(model, dm)
    elapsed = time.perf_counter() - t0

    g_hist = hist.g_loss_epochs
    d_hist = hist.d_loss_epochs
    final_g = g_hist[-1] if g_hist else float("nan")
    final_d = d_hist[-1] if d_hist else float("nan")
    k = min(3, len(g_hist))
    mean_g3 = sum(g_hist[-k:]) / k if k else float("nan")
    mean_d3 = sum(d_hist[-k:]) / k if k else float("nan")

    grid_path = GRID_DIR / f"{experiment_id}.png"
    save_sample_grid(model, grid_path)

    record = {
        "experiment_id": experiment_id,
        "dataset": dataset,
        "max_epochs": max_epochs,
        "batch_size": BATCH_MNIST if dataset == "mnist" else BATCH_FLOWERS,
        "seconds": round(elapsed, 2),
        "hardware_note": GPU_NAME,
        "final_g_loss": round(final_g, 6),
        "final_d_loss": round(final_d, 6),
        "mean_g_loss_last_3_epochs": round(mean_g3, 6),
        "mean_d_loss_last_3_epochs": round(mean_d3, 6),
        "flowers_use_augmentation": flowers_use_augmentation if dataset == "flowers" else None,
        **{f"hparam_{k}": v for k, v in model_kwargs.items()},
        "g_loss_history": [round(x, 6) for x in g_hist],
        "d_loss_history": [round(x, 6) for x in d_hist],
        "sample_grid_png": str(grid_path.as_posix()),
    }
    return record


BASE = dict(
    lr=0.0002,
    label_smooth=0.1,
    latent_dim=100,
    lr_g_mult=2.0,
    lr_d_mult=0.5,
    beta1=0.5,
    d_skip_threshold=0.6,
    d_double_g_threshold=0.4,
)

EXPERIMENTS = [
    ("M1_baseline", "mnist", MNIST_EPOCHS, {**BASE}),
    ("M2_no_label_smooth", "mnist", MNIST_EPOCHS, {**BASE, "label_smooth": 0.0}),
    ("M3_symmetric_lr", "mnist", MNIST_EPOCHS, {**BASE, "lr_g_mult": 1.0, "lr_d_mult": 1.0}),
    ("M4_always_train_D", "mnist", MNIST_EPOCHS, {**BASE, "d_skip_threshold": -1.0}),
    ("M5_latent_50", "mnist", MNIST_EPOCHS, {**BASE, "latent_dim": 50}),
    ("M6_beta1_0.9", "mnist", MNIST_EPOCHS, {**BASE, "beta1": 0.9}),
    ("F1_flowers_baseline", "flowers", FLOWERS_EPOCHS, {**BASE}, True),
    ("F2_flowers_no_aug", "flowers", FLOWERS_EPOCHS, {**BASE}, False),
]

In [ ]:
results = []
for row in EXPERIMENTS:
    if len(row) == 5:
        eid, ds, ep, kw, f_aug = row
        rec = run_experiment(eid, ds, ep, kw, flowers_use_augmentation=f_aug)
    else:
        eid, ds, ep, kw = row
        rec = run_experiment(eid, ds, ep, kw)
    results.append(rec)
    print(f"Done {eid} in {rec['seconds']}s | final g_loss={rec['final_g_loss']} d_loss={rec['final_d_loss']}")

# Tabular summary (drop long histories for CSV)
import pandas as pd

slim = []
for r in results:
    row = {k: v for k, v in r.items() if k not in ("g_loss_history", "d_loss_history")}
    slim.append(row)
df = pd.DataFrame(slim)
csv_path = OUT_DIR / "experiments_summary.csv"
df.to_csv(csv_path, index=False)
print("\nSaved:", csv_path)
display(df)

# Full JSON for your report assistant (copy everything below the line)
print("\n" + "=" * 60)
print("COPY FOR REPORT: EXPERIMENTS_JSON")
print("=" * 60)
print(json.dumps(results, indent=2))
print("=" * 60)

total_sec = sum(r["seconds"] for r in results)
print(f"\nTotal wall time (sum of runs): {total_sec/60:.1f} minutes")

## Optional: plot loss curves (MNIST only)
Uncomment in the next cell if you want a quick multi-line figure for slides.

In [ ]:
# fig, ax = plt.subplots(figsize=(8, 4))
# for r in results:
#     if r["dataset"] != "mnist":
#         continue
#     ax.plot(r["g_loss_history"], label=f"{r['experiment_id']} G")
# ax.set_xlabel("epoch")
# ax.set_ylabel("loss")
# ax.legend(fontsize=7)
# plt.tight_layout()
# plt.savefig(OUT_DIR / "mnist_g_loss_compare.png", dpi=150)
# plt.show()
print("(Optional plotting cell — uncomment to use)")